# 1 - Introducing the API, the JSON format and structure

In this first notebook we read a single resource, a *Person*, from the public
[FAIRDOMHub](https://fairdomhub.org) and look at how SEEK structures the answer.

**What you will learn**

* how a SEEK URL becomes an API call (content negotiation)
* how to send a request with the `requests` library
* the [JSON:API](https://jsonapi.org/) layout that every SEEK resource uses: `data`, `attributes`,
  `relationships`, `links`, `meta`
* how to turn a JSON answer into a table with `pandas`

No login is needed: everything in notebooks 1 to 3 reads **public** data. Writing to SEEK starts in
notebook 4, where the same code gets an API token added.

## 1.1 Import the libraries

* **requests** makes the HTTP calls
* **json** pretty-prints JSON
* **pandas** turns JSON into tables; `json_normalize` flattens a nested dictionary into one row
* **IPython.display** shows images and HTML inside the notebook

In [1]:
import json

import requests
import pandas as pd
from pandas import json_normalize
from IPython.display import Image

## 1.2 Where is the SEEK server?

`base_url` is the address of the SEEK instance, without a trailing slash. Every API route is built by
appending to it. Since we only *read* public data here, we use the FAIRDOMHub directly.

In [2]:
base_url = "https://fairdomhub.org"

## 1.3 Asking for JSON instead of a web page

Open https://fairdomhub.org/people/599 in the browser and you get an HTML page. The API uses the
**same URL**; the only difference is that we tell the server we want JSON. This is called
[content negotiation](https://en.wikipedia.org/wiki/Content_negotiation) and is done with the
`Accept` header:

* `Accept: application/json` - "please answer in JSON"

(You can also see the JSON in the browser by appending `.json` to the URL:
https://fairdomhub.org/people/599.json. SEEK's own media type `application/vnd.api+json` works too.)

We create a `requests.Session` and put the header on it once. A session reuses the connection and
remembers headers, so we do not repeat them in every call. Notebooks 4 to 6 add an
`Authorization` header to this same session; nothing else changes.

In [3]:
session = requests.Session()
session.headers.update({"Accept": "application/json"})

## 1.4 A helper for fetching one resource

SEEK follows RESTful routes: every resource is reachable as `https://<host>/<resource-type>/<id>`,
e.g. `/people/599`, `/projects/60`, `/data_files/1049`. The helper below builds that URL, sends the
request and returns the parsed JSON.

`raise_for_status()` turns an HTTP error (404 not found, 403 not allowed, ...) into a Python exception,
so a typo in the id stops the notebook instead of silently returning garbage. `timeout=30` makes sure a
hanging server does not block the notebook forever.

You will see this helper again in every later notebook.

In [4]:
def json_for_resource(resource_type, resource_id):
    '''GET {base_url}/{resource_type}/{resource_id} and return the parsed JSON.'''
    url = f"{base_url}/{resource_type}/{resource_id}"
    response = session.get(url, timeout=30)
    response.raise_for_status()
    return response.json()

## 1.5 Fetch a person

Every SEEK resource follows the [JSON:API](https://jsonapi.org/) specification, a shared convention that
makes tooling and interoperability easier. The main elements are:

* **data** - the resource itself
* **type** and **id** - what it is and which one
* **attributes** - the metadata (title, description, ...)
* **relationships** - other resources related to this one (projects, institutions, ...)
* **links** - where to find it, usually the link to itself
* **meta** - created/modified timestamps, the API version and the server's base URL

In [5]:
person_id = 599

person = json_for_resource("people", person_id)

print(json.dumps(person, indent=2))

{
  "data": {
    "id": "599",
    "type": "people",
    "attributes": {
      "avatar": "/people/599/avatars/979",
      "title": "Ulrike Wittig",
      "description": "",
      "first_name": "Ulrike",
      "last_name": "Wittig",
      "orcid": "https://orcid.org/0000-0002-9077-5664",
      "mbox_sha1sum": "a3f3fb1bad56371ed5806799679edd8a271c4a3a",
      "expertise": [
        "Biochemistry",
        "Curation",
        "Data Management",
        "Databases"
      ],
      "tools": [
        "SABIO-RK",
        "SEEK"
      ]
    },
    "relationships": {
      "projects": {
        "data": [
          {
            "id": "35",
            "type": "projects"
          },
          {
            "id": "49",
            "type": "projects"
          },
          {
            "id": "50",
            "type": "projects"
          },
          {
            "id": "54",
            "type": "projects"
          },
          {
            "id": "91",
            "type": "projects"
          

Look at the top-level keys first, then at the two that matter most: `type`/`id` and `attributes`.

In [6]:
print("top-level keys :", list(person.keys()))
print("data keys      :", list(person["data"].keys()))
print("type / id      :", person["data"]["type"], "/", person["data"]["id"])
print("attribute keys :", list(person["data"]["attributes"].keys()))

top-level keys : ['data', 'jsonapi']
data keys      : ['id', 'type', 'attributes', 'relationships', 'links', 'meta']
type / id      : people / 599
attribute keys : ['avatar', 'title', 'description', 'first_name', 'last_name', 'orcid', 'mbox_sha1sum', 'expertise', 'tools']


## 1.6 From JSON to a table

`json_normalize` flattens the `attributes` dictionary into a one-row `DataFrame`. Nested keys become
dotted column names. Transposing with `.T` shows one attribute per line, which is easier to read for a
single record.

In [7]:
attributes = json_normalize(person["data"]["attributes"])

attributes.T

,0
avatar,/people/599/avatars/979
title,Ulrike Wittig
description,
first_name,Ulrike
last_name,Wittig
orcid,https://orcid.org/0000-0002-9077-5664
mbox_sha1sum,a3f3fb1bad56371ed5806799679edd8a271c4a3a
expertise,"[Biochemistry, Curation, Data Management, Data..."
tools,"[SABIO-RK, SEEK]"


Individual attributes are ordinary Python values. `tools` is a list of the tools this person has listed
in their profile:

In [8]:
person["data"]["attributes"]["tools"]

['SABIO-RK', 'SEEK']

## 1.7 Relationships

`relationships` lists the other resources this person is connected to. Each entry is just a `type`
and an `id`, i.e. exactly what `json_for_resource` needs to fetch it. Notebook 2 follows these links.

In [9]:
for name, rel in person["data"]["relationships"].items():
    print(f"{name:25} {len(rel['data'])} item(s)")

projects                  30 item(s)
institutions              1 item(s)
investigations            1 item(s)
studies                   0 item(s)
assays                    0 item(s)
data_files                7 item(s)
models                    0 item(s)
sops                      0 item(s)
publications              119 item(s)
presentations             25 item(s)
events                    6 item(s)
documents                 4 item(s)
workflows                 0 item(s)
collections               0 item(s)


## 1.8 Files behind the API

Some attributes are paths rather than values. `avatar` is the path of the person's profile picture,
relative to `base_url`. The same idea applies to downloads in notebook 3.

In [10]:
avatar_url = base_url + person["data"]["attributes"]["avatar"]
print(avatar_url)

Image(url=avatar_url)

https://fairdomhub.org/people/599/avatars/979


## 1.9 Exercises

1. Change `person_id` to another number. What happens with an id that does not exist? Read the exception.
2. Fetch `projects` with id 60 and print its `title`. Which relationship types does a project have that
   a person does not?
3. Replace the `Accept` header with `text/csv` and fetch the person again. What does SEEK answer, and why?